# Programming in Java – Assignment 3 (CO3)
## Complete Colab-ready notebook

This notebook contains all seven assignment tasks. It creates the Java source files and sample inputs, compiles the programs, and provides commands to run each task.

**How to use in Google Colab**
1. Upload this `.ipynb` notebook to Google Colab.
2. Run the cells from top to bottom.
3. The setup cell installs Java if needed and creates the complete `CSA09-192424008-Assignment3` project folder.
4. Run the task cells to compile and execute each program.
5. Download the generated project ZIP from the final cell and upload the extracted files to GitHub.

The examples use Java 17 or later. Thread scheduling order and elapsed times are machine-dependent; record your own outputs.

## 1. Prepare the Java environment

In [ ]:
import os, subprocess, textwrap, zipfile, shutil
from pathlib import Path

# Colab usually provides Java; install OpenJDK 17 if javac is unavailable.
check = subprocess.run(["bash", "-lc", "javac -version"], capture_output=True, text=True)
if check.returncode != 0:
    subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y openjdk-17-jdk-headless -qq"], check=True)

print(subprocess.run(["javac", "-version"], capture_output=True, text=True).stdout or
      subprocess.run(["javac", "-version"], capture_output=True, text=True).stderr)

PROJECT = Path("/content/CSA09-192424008-Assignment3")
PROJECT.mkdir(parents=True, exist_ok=True)

def write_file(relative_path, content):
    path = PROJECT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print("Created:", relative_path)

## Task 1 – ATM Exception Handling

The following cell writes the complete Java source and sample input files into `Task01_ATM/`.

In [ ]:
write_file('Task01_ATM/ATMSimulation.java', 'import java.util.HashMap;\nimport java.util.InputMismatchException;\nimport java.util.Map;\nimport java.util.Scanner;\n\nclass InsufficientFundsException extends Exception {\n    InsufficientFundsException(String message) { super(message); }\n}\nclass AccountNotFoundException extends Exception {\n    AccountNotFoundException(String message) { super(message); }\n}\n\npublic class ATMSimulation {\n    private static final Map<Integer, Double> accounts = new HashMap<>();\n    static {\n        accounts.put(1001, 5000.0);\n        accounts.put(1002, 1200.0);\n    }\n\n    static void withdraw(int account, double amount)\n            throws AccountNotFoundException, InsufficientFundsException {\n        if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");\n        Double balance = accounts.get(account);\n        if (balance == null) throw new AccountNotFoundException("Account not found");\n        if (amount > balance)\n            throw new InsufficientFundsException("Insufficient account balance");\n        accounts.put(account, balance - amount);\n        System.out.println("Please collect cash: Rs." + amount);\n        System.out.println("Remaining balance: Rs." + accounts.get(account));\n    }\n\n    public static void main(String[] args) {\n        System.out.println("ATM session started");\n        try (Scanner sc = new Scanner(System.in)) {\n            System.out.print("Enter account number (1001 or 1002): ");\n            int account = sc.nextInt();\n            System.out.print("Enter withdrawal amount: ");\n            double amount = sc.nextDouble();\n\n            try {\n                withdraw(account, amount);\n            } catch (AccountNotFoundException | InsufficientFundsException e) {\n                System.out.println("Transaction declined: " + e.getMessage());\n            }\n        } catch (InputMismatchException e) {\n            System.out.println("Invalid input: enter numeric values only.");\n        } catch (IllegalArgumentException e) {\n            System.out.println("Invalid withdrawal: " + e.getMessage());\n        } catch (Exception e) {\n            System.out.println("ATM service error: " + e.getMessage());\n        } finally {\n            System.out.println("ATM session closed. Please take your card.");\n        }\n    }\n}\n')

## Task 2 – User-defined Exceptions and Propagation

The following cell writes the complete Java source and sample input files into `Task02_Exceptions/`.

In [ ]:
write_file('Task02_Exceptions/StorageDemo.java', 'import java.io.IOException;\n\nclass ExamException extends Exception {\n    ExamException(String message) { super(message); }\n    ExamException(String message, Throwable cause) { super(message, cause); }\n}\nclass ExamWindowClosedException extends ExamException {\n    ExamWindowClosedException(String message) { super(message); }\n}\nclass StudentNotRegisteredException extends ExamException {\n    StudentNotRegisteredException(String message) { super(message); }\n}\nclass SubmissionStorageException extends ExamException {\n    SubmissionStorageException(String message, Throwable cause) {\n        super(message, cause);\n    }\n}\nclass InvalidSubmissionRequestException extends RuntimeException {\n    InvalidSubmissionRequestException(String message) { super(message); }\n}\n\nclass Storage {\n    void save(String fileName, long sizeKb) throws SubmissionStorageException {\n        try {\n            if (sizeKb > 500) throw new IOException("Disk quota exceeded");\n            System.out.println("Saved " + fileName);\n        } catch (IOException e) {\n            throw new SubmissionStorageException("Could not store " + fileName, e);\n        }\n    }\n}\nclass ExamService {\n    private final Storage storage = new Storage();\n\n    void submit(String studentId, boolean registered, boolean examOpen,\n                String fileName, long sizeKb) throws ExamException {\n        if (studentId == null || studentId.trim().isEmpty())\n            throw new InvalidSubmissionRequestException("Student ID is required");\n        if (!examOpen) throw new ExamWindowClosedException("Exam window is closed");\n        if (!registered)\n            throw new StudentNotRegisteredException("Student is not registered");\n        storage.save(fileName, sizeKb);\n    }\n}\npublic class StorageDemo {\n    public static void main(String[] args) {\n        ExamService service = new ExamService();\n        try {\n            service.submit("STU001", true, true, "answer01.pdf", 120);\n            service.submit("STU002", true, true, "answer02.pdf", 800);\n        } catch (ExamException e) {\n            System.out.println("Portal message: " + e.getMessage());\n            if (e.getCause() != null)\n                System.out.println("Original cause: " + e.getCause().getMessage());\n            e.printStackTrace();\n        } catch (InvalidSubmissionRequestException e) {\n            System.out.println("Invalid request: " + e.getMessage());\n        }\n    }\n}\n')

## Task 3 – Resource Safety and Try-with-resources

The following cell writes the complete Java source and sample input files into `Task03_Resources/`.

In [ ]:
write_file('Task03_Resources/ManualSalesReport.java', 'import java.io.*;\n\npublic class ManualSalesReport {\n    public static void generate(String[] files, String output) {\n        BufferedWriter writer = null;\n        try {\n            writer = new BufferedWriter(new FileWriter(output));\n            long total = 0;\n            for (String file : files) {\n                BufferedReader reader = null;\n                try {\n                    reader = new BufferedReader(new FileReader(file));\n                    String line;\n                    while ((line = reader.readLine()) != null)\n                        total += Long.parseLong(line.trim());\n                } catch (IOException | NumberFormatException e) {\n                    System.err.println("Failed file " + file + ": " + e.getMessage());\n                    throw new RuntimeException("Report incomplete", e);\n                } finally {\n                    if (reader != null) {\n                        try { reader.close(); }\n                        catch (IOException closeError) {\n                            System.err.println("Close failed: " + closeError.getMessage());\n                        }\n                    }\n                }\n            }\n            writer.write("TOTAL=" + total);\n        } catch (IOException | RuntimeException e) {\n            System.err.println("Report generation failed: " + e.getMessage());\n        } finally {\n            if (writer != null) {\n                try { writer.close(); }\n                catch (IOException e) {\n                    System.err.println("Output close failed: " + e.getMessage());\n                }\n            }\n        }\n    }\n\n    public static void main(String[] args) {\n        String[] files = args.length > 0 ? args :\n            new String[]{"test-files/store1.txt", "test-files/store2.txt"};\n        generate(files, "manual-summary.txt");\n        System.out.println("Manual report attempt finished.");\n    }\n}\n')

In [ ]:
write_file('Task03_Resources/SafeSalesReport.java', 'import java.io.*;\n\npublic class SafeSalesReport {\n    public static void generate(String[] files, String output) {\n        long total = 0;\n        try (BufferedWriter writer = new BufferedWriter(new FileWriter(output))) {\n            for (String file : files) {\n                System.out.println("Processing: " + file);\n                try (BufferedReader reader = new BufferedReader(new FileReader(file))) {\n                    String line;\n                    while ((line = reader.readLine()) != null)\n                        total += Long.parseLong(line.trim());\n                } catch (IOException | NumberFormatException e) {\n                    throw new RuntimeException("Could not process " + file, e);\n                }\n            }\n            writer.write("TOTAL=" + total);\n        } catch (IOException | RuntimeException e) {\n            System.err.println("Report failed: " + e.getMessage());\n        }\n    }\n\n    public static void main(String[] args) {\n        String[] files = args.length > 0 ? args :\n            new String[]{"test-files/store1.txt", "test-files/store2.txt"};\n        generate(files, "safe-summary.txt");\n        System.out.println("Safe report attempt finished.");\n    }\n}\n')

In [ ]:
write_file('Task03_Resources/test-files/store1.txt', '1200\n800\n450\n')

In [ ]:
write_file('Task03_Resources/test-files/store2.txt', '950\n600\n')

## Task 4 – Thread Creation and Life Cycle

The following cell writes the complete Java source and sample input files into `Task04_Threads/`.

In [ ]:
write_file('Task04_Threads/RunnableStations.java', 'class ScannerTask implements Runnable {\n    private final int station, scripts;\n    ScannerTask(int station, int scripts) {\n        this.station = station; this.scripts = scripts;\n    }\n    public void run() {\n        try {\n            for (int i = 0; i < scripts; i++) Thread.sleep(10);\n            System.out.println("Station " + station + " scanned " + scripts);\n        } catch (InterruptedException e) {\n            Thread.currentThread().interrupt();\n        }\n    }\n}\npublic class RunnableStations {\n    public static void main(String[] args) throws InterruptedException {\n        Thread[] stations = new Thread[4];\n        for (int i = 0; i < 4; i++) {\n            stations[i] = new Thread(new ScannerTask(i + 1, 5));\n            stations[i].start();\n        }\n        for (Thread t : stations) t.join();\n        System.out.println("All stations completed.");\n    }\n}\n')

In [ ]:
write_file('Task04_Threads/ThreadExperiments.java', 'public class ThreadExperiments {\n    public static void main(String[] args) throws InterruptedException {\n        Thread high = new Thread(() -> work("HIGH"));\n        Thread low = new Thread(() -> work("LOW"));\n        high.setPriority(Thread.MAX_PRIORITY);\n        low.setPriority(Thread.MIN_PRIORITY);\n\n        high.start();\n        low.start();\n        System.out.println("High thread alive? " + high.isAlive());\n        System.out.println("Low thread alive? " + low.isAlive());\n        high.join();\n        low.join();\n\n        Thread daemon = new Thread(() -> {\n            try {\n                Thread.sleep(1000);\n                System.out.println("Daemon task finished");\n            } catch (InterruptedException e) {\n                Thread.currentThread().interrupt();\n            }\n        });\n        daemon.setDaemon(true); // Must be set before start().\n        daemon.start();\n        System.out.println("Main completed; daemon may be stopped when JVM exits.");\n    }\n\n    private static void work(String label) {\n        for (int i = 1; i <= 5; i++)\n            System.out.println(label + " thread step " + i);\n    }\n}\n')

In [ ]:
write_file('Task04_Threads/ThreadStations.java', 'class ScannerStationThread extends Thread {\n    private final int station, scripts;\n    ScannerStationThread(int station, int scripts) {\n        this.station = station; this.scripts = scripts;\n    }\n    public void run() {\n        try {\n            for (int i = 0; i < scripts; i++) Thread.sleep(10);\n            System.out.println("Station " + station + " scanned " + scripts);\n        } catch (InterruptedException e) {\n            Thread.currentThread().interrupt();\n        }\n    }\n}\npublic class ThreadStations {\n    public static void main(String[] args) throws InterruptedException {\n        Thread[] stations = new Thread[4];\n        for (int i = 0; i < 4; i++) {\n            stations[i] = new ScannerStationThread(i + 1, 5);\n            stations[i].start();\n        }\n        for (Thread t : stations) t.join();\n        System.out.println("All stations completed.");\n    }\n}\n')

## Task 5 – Race Conditions and Synchronization

The following cell writes the complete Java source and sample input files into `Task05_Synchronization/`.

In [ ]:
write_file('Task05_Synchronization/BookingRaceExperiment.java', 'public class BookingRaceExperiment {\n    public static void main(String[] args) throws InterruptedException {\n        for (int trial = 1; trial <= 10; trial++) {\n            SeatBooking screen = new SeatBooking(10);\n            Thread[] users = new Thread[100];\n            long start = System.nanoTime();\n\n            for (int i = 0; i < users.length; i++) {\n                final int seat = i % 10;\n                users[i] = new Thread(() -> screen.bookUnsafe(seat));\n                users[i].start();\n            }\n            for (Thread t : users) t.join();\n\n            long elapsed = System.nanoTime() - start;\n            System.out.printf("Trial %d: unsafe seats sold=%d, elapsed=%.3f ms%n",\n                    trial, screen.getBookingsMade(), elapsed / 1_000_000.0);\n        }\n    }\n}\n')

In [ ]:
write_file('Task05_Synchronization/SeatBooking.java', 'public class SeatBooking {\n    private final boolean[] booked;\n    private int bookingsMade = 0;\n\n    SeatBooking(int seats) { booked = new boolean[seats]; }\n\n    // Intentionally unsafe version to demonstrate a race condition.\n    public boolean bookUnsafe(int seat) {\n        if (!booked[seat]) {\n            Thread.yield();\n            booked[seat] = true;\n            bookingsMade++;\n            return true;\n        }\n        return false;\n    }\n\n    public synchronized boolean book(int seat) {\n        if (booked[seat]) return false;\n        booked[seat] = true;\n        bookingsMade++;\n        return true;\n    }\n\n    public boolean bookWithBlock(int seat) {\n        synchronized (this) {\n            if (booked[seat]) return false;\n            booked[seat] = true;\n            bookingsMade++;\n            return true;\n        }\n    }\n\n    public int getBookingsMade() { return bookingsMade; }\n\n    public static void main(String[] args) throws InterruptedException {\n        runTrial("Synchronized method", 1);\n        runTrial("Synchronized block", 2);\n    }\n\n    private static void runTrial(String label, int mode) throws InterruptedException {\n        SeatBooking screen = new SeatBooking(10);\n        Thread[] users = new Thread[100];\n        for (int i = 0; i < users.length; i++) {\n            final int seat = i % 10;\n            users[i] = new Thread(() -> {\n                if (mode == 1) screen.book(seat);\n                else screen.bookWithBlock(seat);\n            });\n            users[i].start();\n        }\n        for (Thread t : users) t.join();\n        System.out.println(label + " – seats sold: " + screen.getBookingsMade());\n    }\n}\n')

## Task 6 – Deadlock and Producer–Consumer

The following cell writes the complete Java source and sample input files into `Task06_Deadlock_ProducerConsumer/`.

In [ ]:
write_file('Task06_Deadlock_ProducerConsumer/TokenCounter.java', 'public class TokenCounter {\n    private final int[] buffer = new int[5];\n    private int count = 0, in = 0, out = 0;\n\n    public synchronized void put(int token) throws InterruptedException {\n        while (count == buffer.length) wait();\n        buffer[in] = token;\n        in = (in + 1) % buffer.length;\n        count++;\n        notifyAll();\n    }\n\n    public synchronized int take() throws InterruptedException {\n        while (count == 0) wait();\n        int token = buffer[out];\n        out = (out + 1) % buffer.length;\n        count--;\n        notifyAll();\n        return token;\n    }\n\n    public static void main(String[] args) throws InterruptedException {\n        TokenCounter counter = new TokenCounter();\n        int[] collected = new int[2];\n\n        Thread cook = new Thread(() -> {\n            try {\n                for (int token = 1; token <= 20; token++) counter.put(token);\n            } catch (InterruptedException e) {\n                Thread.currentThread().interrupt();\n            }\n        });\n\n        Thread[] students = new Thread[2];\n        for (int s = 0; s < 2; s++) {\n            final int id = s;\n            students[s] = new Thread(() -> {\n                try {\n                    for (int k = 0; k < 10; k++) collected[id] += counter.take();\n                } catch (InterruptedException e) {\n                    Thread.currentThread().interrupt();\n                }\n            });\n        }\n\n        cook.start();\n        for (Thread student : students) student.start();\n        cook.join();\n        for (Thread student : students) student.join();\n\n        System.out.println("Sum of tokens collected: "\n                + (collected[0] + collected[1]) + " (expected 210)");\n    }\n}\n')

In [ ]:
write_file('Task06_Deadlock_ProducerConsumer/Transfers.java', 'class Wallet {\n    final int id;\n    int balance;\n    Wallet(int id, int balance) { this.id = id; this.balance = balance; }\n}\npublic class Transfers {\n    // Safe strategy: every transfer locks wallets in ascending ID order.\n    static void transferSafe(Wallet from, Wallet to, int amount) {\n        if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");\n        Wallet first = from.id < to.id ? from : to;\n        Wallet second = from.id < to.id ? to : from;\n        synchronized (first) {\n            synchronized (second) {\n                if (from.balance < amount)\n                    throw new IllegalArgumentException("Insufficient balance");\n                from.balance -= amount;\n                to.balance += amount;\n            }\n        }\n    }\n\n    public static void main(String[] args) {\n        Wallet a = new Wallet(1, 1000);\n        Wallet b = new Wallet(2, 500);\n        transferSafe(a, b, 200);\n        System.out.println("Wallet A: " + a.balance);\n        System.out.println("Wallet B: " + b.balance);\n        System.out.println("Total: " + (a.balance + b.balance));\n    }\n}\n')

## Task 7 – Concurrent Fault-tolerant Result Processor

The following cell writes the complete Java source and sample input files into `Task07_ConcurrentProcessor/`.

In [ ]:
write_file('Task07_ConcurrentProcessor/ConcurrentResultProcessor.java', 'import java.io.BufferedWriter;\nimport java.io.IOException;\nimport java.nio.file.Files;\nimport java.nio.file.Path;\nimport java.nio.file.Paths;\nimport java.util.ArrayList;\nimport java.util.List;\nimport java.util.concurrent.Callable;\nimport java.util.concurrent.ConcurrentLinkedQueue;\nimport java.util.concurrent.ExecutionException;\nimport java.util.concurrent.ExecutorService;\nimport java.util.concurrent.Executors;\nimport java.util.concurrent.Future;\nimport java.util.concurrent.TimeUnit;\n\nclass FileResult {\n    final String file, studentId, status, message;\n    final int total;\n\n    FileResult(String file, String studentId, String status, int total, String message) {\n        this.file = file;\n        this.studentId = studentId;\n        this.status = status;\n        this.total = total;\n        this.message = message;\n    }\n\n    public String toString() {\n        return file + " | " + status + " | " + studentId + " | " + total + " | " + message;\n    }\n}\n\nclass ScoreTask implements Callable<FileResult> {\n    private final Path path;\n\n    ScoreTask(Path path) { this.path = path; }\n\n    public FileResult call() {\n        try {\n            List<String> lines = Files.readAllLines(path);\n            if (lines.size() < 2)\n                throw new IllegalArgumentException("Missing score data");\n\n            String studentId = lines.get(0).trim();\n            if (studentId.isEmpty())\n                throw new IllegalArgumentException("Missing student ID");\n\n            int total = 0;\n            for (int i = 1; i < lines.size(); i++) {\n                int mark = Integer.parseInt(lines.get(i).trim());\n                if (mark < 0 || mark > 100)\n                    throw new IllegalArgumentException("Score outside 0..100");\n                total += mark;\n            }\n            return new FileResult(path.getFileName().toString(),\n                    studentId, "SUCCESS", total, "validated");\n        } catch (IOException | RuntimeException e) {\n            return new FileResult(path.getFileName().toString(),\n                    "-", "FAILURE", 0, e.getMessage());\n        }\n    }\n}\n\npublic class ConcurrentResultProcessor {\n    static void run(Path folder, int poolSize) throws Exception {\n        Path manifest = folder.resolve("manifest.txt");\n        List<String> names = Files.readAllLines(manifest);\n        List<Path> expectedFiles = new ArrayList<>();\n        for (String name : names) {\n            String trimmed = name.trim();\n            if (!trimmed.isEmpty()) expectedFiles.add(folder.resolve(trimmed));\n        }\n\n        ExecutorService pool = Executors.newFixedThreadPool(poolSize);\n        ConcurrentLinkedQueue<FileResult> results = new ConcurrentLinkedQueue<>();\n        List<Future<FileResult>> futures = new ArrayList<>();\n        long start = System.nanoTime();\n\n        try {\n            for (Path file : expectedFiles)\n                futures.add(pool.submit(new ScoreTask(file)));\n\n            for (Future<FileResult> future : futures) {\n                try {\n                    results.add(future.get());\n                } catch (ExecutionException e) {\n                    results.add(new FileResult("unknown", "-", "FAILURE",\n                            0, String.valueOf(e.getCause())));\n                }\n            }\n        } finally {\n            pool.shutdown();\n            if (!pool.awaitTermination(1, TimeUnit.MINUTES))\n                pool.shutdownNow();\n        }\n\n        long elapsed = System.nanoTime() - start;\n        long success = results.stream().filter(r -> r.status.equals("SUCCESS")).count();\n        long failure = results.size() - success;\n\n        System.out.println("Files expected: " + expectedFiles.size());\n        System.out.println("Success: " + success);\n        System.out.println("Failure: " + failure);\n        System.out.println("Total outcomes: " + results.size());\n        System.out.printf("Elapsed: %.3f ms (pool=%d)%n", elapsed / 1e6, poolSize);\n\n        if (results.size() != expectedFiles.size())\n            throw new IllegalStateException("Incomplete summary");\n\n        Path summary = folder.resolve("summary.txt");\n        try (BufferedWriter writer = Files.newBufferedWriter(summary)) {\n            for (FileResult result : results)\n                writer.write(result + System.lineSeparator());\n        }\n    }\n\n    public static void main(String[] args) throws Exception {\n        Path folder = Paths.get(args.length > 0 ? args[0] : "scores");\n        int poolSize = args.length > 1 ? Integer.parseInt(args[1]) : 4;\n        if (poolSize < 1) throw new IllegalArgumentException("Pool size must be >= 1");\n        run(folder, poolSize);\n    }\n}\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/manifest.txt', 'S001.txt\nS002.txt\nS003.txt\nS004.txt\nS005_missing.txt\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/scores/S001.txt', 'STU001\n80\n75\n90\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/scores/S002.txt', 'STU002\n60\n70\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/scores/S003.txt', 'STU003\n80\nBAD\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/scores/S004.txt', 'STU004\n120\n')

## 8. Add repository README and output folders

In [ ]:
write_file('README.md', '# Java Assignment 3 – CO3\n\nRepository: `CSA09-192424008-Assignment3`\n\nThis repository contains Java source files and sample input files for all seven Assignment 3 tasks.\n\n## Requirements\n- JDK 17 or later recommended.\n- Open a terminal in the relevant task folder.\n- Compile the source file with `javac`.\n- Run the class with `java`.\n\n## Folder structure\n\n- `Task01_ATM` – ATM exception handling and custom checked exceptions.\n- `Task02_Exceptions` – user-defined exceptions, exception propagation and cause chaining.\n- `Task03_Resources` – manual cleanup and try-with-resources report generation.\n- `Task04_Threads` – Thread, Runnable, priorities, daemon and lifecycle experiments.\n- `Task05_Synchronization` – unsafe race condition and synchronized booking.\n- `Task06_Deadlock_ProducerConsumer` – safe wallet transfers and bounded producer-consumer buffer.\n- `Task07_ConcurrentProcessor` – ExecutorService, Callable, Future and manifest-based processing.\n\n## Compile and run\n\nRun commands from inside each task folder.\n\n### Task 1 – ATM\n```bash\njavac ATMSimulation.java\njava ATMSimulation\n```\nSample valid account numbers: `1001` (balance Rs. 5000) and `1002` (balance Rs. 1200).\n\n### Task 2 – Exception propagation\n```bash\njavac StorageDemo.java\njava StorageDemo\n```\nThe second sample submission is intentionally larger than 500 KB to demonstrate exception chaining.\n\n### Task 3 – Resource handling\n```bash\njavac ManualSalesReport.java SafeSalesReport.java\njava ManualSalesReport\njava SafeSalesReport\n```\nBoth programs read `test-files/store1.txt` and `test-files/store2.txt` relative to the current task folder.\n\n### Task 4 – Threads\n```bash\njavac ThreadStations.java RunnableStations.java ThreadExperiments.java\njava ThreadStations\njava RunnableStations\njava ThreadExperiments\n```\nThread completion order can differ between runs. Priority is only a scheduling hint.\n\n### Task 5 – Seat booking\n```bash\njavac SeatBooking.java BookingRaceExperiment.java\njava SeatBooking\njava BookingRaceExperiment\n```\nThe unsafe experiment is timing-dependent. Synchronized booking should never sell more than ten unique seats.\n\n### Task 6 – Deadlock and producer-consumer\n```bash\njavac Transfers.java TokenCounter.java\njava Transfers\njava TokenCounter\n```\nThe wallet example uses a consistent lock order. The token counter uses a bounded buffer of capacity five.\n\n### Task 7 – Concurrent result processor\nRun from inside `Task07_ConcurrentProcessor`:\n```bash\njavac ConcurrentResultProcessor.java\njava ConcurrentResultProcessor . 4\n```\nThe program reads expected filenames from `manifest.txt`. A missing or malformed file is recorded as a failure without stopping the remaining tasks.\n\nRun pool-size experiments:\n```bash\njava ConcurrentResultProcessor . 1\njava ConcurrentResultProcessor . 2\njava ConcurrentResultProcessor . 4\njava ConcurrentResultProcessor . 8\n```\nRecord actual elapsed times in `measurements/`. Do not invent timing values.\n\n## Important notes\n- `Task07_ConcurrentProcessor/scores/S005_missing.txt` is intentionally absent. It appears in the manifest so the processor can demonstrate missing-file handling.\n- Generated summary files and console output screenshots are not pre-filled. Run the programs and add your own outputs to the `outputs/` or `measurements/` folders.\n- The programs are educational demonstrations and are not production banking, examination, cinema, or clinical systems.\n')

In [ ]:
write_file('Task01_ATM/outputs/README.txt', 'Add your own console output screenshots or captured results here.\n')

In [ ]:
write_file('Task02_Exceptions/outputs/README.txt', 'Add your own console output screenshots or captured results here.\n')

In [ ]:
write_file('Task04_Threads/outputs/README.txt', 'Add your own console output screenshots or captured results here.\n')

In [ ]:
write_file('Task05_Synchronization/outputs/README.txt', 'Add your own console output screenshots or captured results here.\n')

In [ ]:
write_file('Task06_Deadlock_ProducerConsumer/outputs/README.txt', 'Add your own console output screenshots or captured results here.\n')

In [ ]:
write_file('Task07_ConcurrentProcessor/measurements/README.txt', 'Record measured run times for pool sizes 1, 2, 4 and 8 here. Use identical input data for each run.\n')

## 9. Compile all Java programs

In [ ]:
task_folders = [
    "Task01_ATM", "Task02_Exceptions", "Task03_Resources",
    "Task04_Threads", "Task05_Synchronization",
    "Task06_Deadlock_ProducerConsumer", "Task07_ConcurrentProcessor"
]

for folder in task_folders:
    java_files = sorted(str(p) for p in (PROJECT/folder).glob("*.java"))
    result = subprocess.run(["javac"] + java_files, capture_output=True, text=True)
    print(f"\n[{folder}]")
    print("Compilation:", "SUCCESS" if result.returncode == 0 else "FAILED")
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Compilation failed in {folder}")

## Task 1 – Run ATM

ATM expects account number and amount. Example input: 1001 then 500.

In [ ]:
result = subprocess.run(['bash','-lc', 'printf \'1001\\n500\\n\' | java -cp "$PROJECT/Task01_ATM" ATMSimulation'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 2 – Run exception propagation demo

Shows a successful save followed by a storage exception whose original IOException cause is retained.

In [ ]:
result = subprocess.run(['bash','-lc', 'java -cp "$PROJECT/Task02_Exceptions" StorageDemo'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 3 – Run resource-safety programs

Runs both manual-close and try-with-resources versions using the sample store files.

In [ ]:
result = subprocess.run(['bash','-lc', 'cd "$PROJECT/Task03_Resources" && java ManualSalesReport && java SafeSalesReport && echo \'--- Generated report files ---\' && cat manual-summary.txt safe-summary.txt'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 4 – Run thread demonstrations

The order of station messages may vary because thread scheduling is nondeterministic.

In [ ]:
result = subprocess.run(['bash','-lc', 'java -cp "$PROJECT/Task04_Threads" ThreadStations && java -cp "$PROJECT/Task04_Threads" RunnableStations && java -cp "$PROJECT/Task04_Threads" ThreadExperiments'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 5 – Run synchronized booking and race experiment

Synchronized versions should sell no more than ten seats. The unsafe experiment is timing-dependent.

In [ ]:
result = subprocess.run(['bash','-lc', 'java -cp "$PROJECT/Task05_Synchronization" SeatBooking && java -cp "$PROJECT/Task05_Synchronization" BookingRaceExperiment'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 6 – Run wallet transfer and producer-consumer

The token counter should print a total of 210.

In [ ]:
result = subprocess.run(['bash','-lc', 'java -cp "$PROJECT/Task06_Deadlock_ProducerConsumer" Transfers && java -cp "$PROJECT/Task06_Deadlock_ProducerConsumer" TokenCounter'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## Task 7 – Run concurrent result processor

The manifest includes an intentionally missing file and corrupt/out-of-range sample files. All expected files should receive a success/failure outcome.

In [ ]:
result = subprocess.run(['bash','-lc', 'cd "$PROJECT/Task07_ConcurrentProcessor" && java ConcurrentResultProcessor . 4 && echo \'--- Summary file ---\' && cat summary.txt'], cwd='/content', capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
if result.returncode != 0: raise RuntimeError('Execution failed')

## 10. Benchmark the concurrent processor

Run the same input set with pool sizes 1, 2, 4 and 8. The elapsed time is specific to the current Colab runtime. Repeat each configuration several times and record your actual values in your report.

In [ ]:
for pool_size in [1, 2, 4, 8]:
    result = subprocess.run(
        ["java", "ConcurrentResultProcessor", ".", str(pool_size)],
        cwd=PROJECT/"Task07_ConcurrentProcessor",
        capture_output=True, text=True
    )
    print(f"\n--- Pool size {pool_size} ---")
    print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode != 0: raise RuntimeError("Benchmark run failed")

## 11. Create the GitHub upload ZIP

This packages the generated Java files, sample inputs, README and folder structure. The `.class` files and generated summary outputs are excluded so the repository contains clean source code.

In [ ]:
zip_path = Path("/content/CSA09-192424008-Assignment3.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in PROJECT.rglob("*"):
        if path.is_file() and path.suffix != ".class" and path.name not in {
            "manual-summary.txt", "safe-summary.txt", "summary.txt"
        }:
            zf.write(path, path.relative_to(PROJECT.parent))

print("Created:", zip_path)
print("ZIP size:", zip_path.stat().st_size, "bytes")
print("Java files:", len(list(PROJECT.rglob("*.java"))))

# In Google Colab, uncomment the next two lines to download automatically:
from google.colab import files
files.download(str(zip_path))